# L5 — Unire tabelle (SOLUZIONI)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrealorenzon/SIMA2026/blob/soluzioni/L05_unire/L05_soluzioni.ipynb)

## Promemoria

- **Shift + Invio** esegue una cella. **File → Salva una copia in Drive** per non perdere il lavoro.
- Per ogni passo trovi 🔎 **cosa cercare**. Gemini (✨) è disponibile, ma prova prima a capire cosa ti serve.
- ⏱️ Bloccato da più di 10 minuti? Chiedi.
- Se fai un pasticcio: **Runtime → Riavvia ed esegui tutto**.

## Contesto

Il dataset delle imprese dice in che regione si trova ogni impresa, ma non dice niente delle regioni: macroarea, popolazione. Queste informazioni sono in un altro file, `regioni.xlsx`. Per usarle insieme bisogna **unire** le due tabelle, come fa `CERCA.VERT` in Excel.

**La regola di oggi:** dopo ogni unione, conta le righe. Un'unione sbagliata non dà quasi mai errori.

## 0. Preparazione

Esegui questa cella così com'è.

In [ ]:
import pandas as pd

df = pd.read_excel("https://raw.githubusercontent.com/andrealorenzon/SIMA2026/main/dati/imprese_pulito.xlsx")
regioni = pd.read_excel("https://raw.githubusercontent.com/andrealorenzon/SIMA2026/main/dati/regioni.xlsx")
print(df.shape)

## Parte 1 — L'unione

### 1. Guarda la seconda tabella

Mostra la tabella `regioni`: quante righe ha? Quale colonna ha in comune con `df`?

🔎 *Da cercare:* `pandas head` · `pandas shape`

In [ ]:
print(regioni.shape)
regioni

### 2. Unisci

Unisci `df` e `regioni` usando la colonna in comune, e salva il risultato in `unito`. Quante righe ha `unito`? Quante dovrebbe averne?

🔎 *Da cercare:* `pandas merge on`

In [ ]:
unito = df.merge(regioni, on="regione")
unito.shape

*Risposta:* 2.240 invece di 2.500: sono sparite 260 righe, senza nessun errore. Per impostazione predefinita `merge` tiene solo le righe che trovano una corrispondenza in **entrambe** le tabelle.

### 3. Chi è sparito?

Rifai l'unione tenendo **tutte** le righe di `df`, e chiedi a pandas di aggiungere una colonna che dice da quale tabella viene ogni riga. Quali regioni non hanno trovato corrispondenza?

🔎 *Da cercare:* `pandas merge how left` · `pandas merge indicator`

In [ ]:
prova = df.merge(regioni, on="regione", how="left", indicator=True)
print(prova["_merge"].value_counts())
prova[prova["_merge"] == "left_only"]["regione"].value_counts()

*Risposta:* Emilia-Romagna (190 righe) e Trentino-Alto Adige (70): in `regioni.xlsx` sono scritte `Emilia Romagna` (senza trattino) e `Trentino-Alto Adige/Suedtirol`. Per Python sono nomi diversi.

### 4. Correggi le chiavi

Correggi i nomi delle due regioni nella tabella `regioni`, in modo che coincidano con quelli di `df`. Poi rifai l'unione (con `how="left"`) e verifica: 2.500 righe, nessuna macroarea mancante.

🔎 *Da cercare:* `pandas replace`

In [ ]:
regioni["regione"] = regioni["regione"].replace({
    "Emilia Romagna": "Emilia-Romagna",
    "Trentino-Alto Adige/Suedtirol": "Trentino-Alto Adige",
})
unito = df.merge(regioni, on="regione", how="left")
print(unito.shape, unito["macroarea"].isna().sum())

## Parte 2 — Usare l'unione

### 5. Le macroaree

Calcola il fatturato mediano per macroarea.

🔎 *Da cercare:* `pandas groupby median`

In [ ]:
unito.groupby("macroarea")["fatturato"].median().sort_values()

### 6. Quante imprese per abitante?

Vogliamo sapere quante imprese del campione ci sono ogni milione di abitanti, regione per regione.

1. Conta le imprese **diverse** per regione e trasforma il risultato in una tabella.
2. Unisci questa tabella con `regioni`.
3. Calcola imprese ogni milione di abitanti (la popolazione è in migliaia).

🔎 *Da cercare:* `pandas groupby nunique` · `pandas reset_index`

In [ ]:
conteggi = df.groupby("regione")["id_impresa"].nunique().reset_index(name="imprese")
tab = conteggi.merge(regioni, on="regione", how="left")
tab["per_milione"] = tab["imprese"] / tab["popolazione_migliaia"] * 1000
tab.sort_values("per_milione", ascending=False).head()

### 7. Quanti italiani?

Calcola la popolazione totale sommando la colonna `popolazione_migliaia`, prima nella tabella `regioni` e poi nella tabella `unito`. I due numeri sono uguali? Quale è giusto, e perché l'altro è sbagliato?

🔎 *Da cercare:* `pandas sum`

In [ ]:
print(regioni["popolazione_migliaia"].sum())
print(unito["popolazione_migliaia"].sum())

*Risposta:* Giusto quello di `regioni`: circa 59 milioni. In `unito` la popolazione di una regione è ripetuta su **ogni riga** di ogni impresa di quella regione, per ogni anno: sommandola si ottengono quasi 12 miliardi di "italiani". Dopo un'unione, i valori della tabella più piccola sono ripetuti: non si sommano.

## Domanda finale

Scrivi **due regole** che seguirai ogni volta che unisci due tabelle.

*Risposta:* 1) Conta le righe prima e dopo, e controlla i valori mancanti nelle colonne aggiunte; 2) usa `how="left"` e `indicator=True` per vedere cosa non ha trovato corrispondenza. Bonus: non sommare colonne che l'unione ha ripetuto.

---
## ⭐ Bonus (per chi ha finito)

### B1. Un controllo automatico

`merge` può controllare da solo che ogni regione compaia **una sola volta** nella tabella `regioni`. Trova il parametro e usalo.

🔎 *Da cercare:* `pandas merge validate many_to_one`

In [ ]:
unito = df.merge(regioni, on="regione", how="left", validate="many_to_one")
unito.shape

### B2. Mettere in fila

Unire non vuol dire solo affiancare colonne: a volte si vogliono mettere **una sotto l'altra** due tabelle con le stesse colonne. Prendi le righe del 2019 e quelle del 2023 come due tabelle separate e rimettile insieme in una sola.

🔎 *Da cercare:* `pandas concat`

In [ ]:
a = df[df["anno"] == 2019]
b2 = df[df["anno"] == 2023]
insieme = pd.concat([a, b2])
print(len(a), len(b2), len(insieme))

### B3. Disegnalo

Disegna un grafico a barre orizzontali delle imprese ogni milione di abitanti, per regione, ordinato.

🔎 *Da cercare:* `pandas plot barh`

In [ ]:
import matplotlib.pyplot as plt
tab.set_index("regione")["per_milione"].sort_values().plot(kind="barh")
plt.show()

---
## Compito a casa

- [Kaggle Learn – Pandas](https://www.kaggle.com/learn/pandas), lezione **Renaming and Combining** (~20 min)
- Facoltativo: [W3Schools – Pandas](https://www.w3schools.com/python/pandas/default.asp), per ripassare